# 05 — AryColBring vs LTR-LGBM: A Comparison

**Objective**: compare AryColBring (collaborative filtering, embedding dot-product) against LTR-LGBM (gradient-boosted learning-to-rank) on Precision@K / Recall@K / NDCG@K, and lay out when each one is the better fit.

**Audience**: anyone deciding which of the two recommenders in this repo to use for a given surface.

> Both models need their own compiled/heavy dependencies to train for real (AryColBring needs the `CLproximity` Cython extension; LTR-LGBM needs `lightgbm`). Neither is available in every environment this notebook might run in, so the comparison below runs on synthetic ranked outputs shaped exactly like what each model actually returns -- this notebook is about the *evaluation methodology and trade-offs*, not about reproducing a training run (see `03_Training_AryColBring.ipynb` for that, on the AryColBring side).

## 1. What each model actually is

| | **AryColBring** | **LTR-LGBM** |
|---|---|---|
| Approach | Collaborative filtering -- learns a `user_embedding`/`item_embedding` per id, scores as dot product + biases (`src/models/arycolbring/inout/approximator.py`) | Learning-to-rank -- gradient-boosted trees over hand/auto-engineered features (`src/models/ltr_lgbm/`), no embeddings |
| Needs at inference time | Just the user/item id (embeddings are looked up) | A full feature row per (user, item) candidate -- price, recency, category match, etc. |
| Cold start | Item embeddings enable item-to-item fallback even for brand-new users (`06_Cold_Start_Handling.ipynb`) | Needs the same features to be computable for the new user/item, which is often the harder problem in practice |
| Typical strength | Captures latent taste patterns purely from interaction history, cheap to serve at scale | Can directly exploit business features (price sensitivity, promotions, recency) that collaborative filtering can't see at all |

## 2. Synthetic ranked output for both models

We simulate what each model would produce for the same 50 users against a 200-item catalog: `true_relevance` is the "ground truth" (held-out purchases), and each model's ranking is generated with a different, plausible error profile -- AryColBring tends to do well on popular/well-connected items (embedding-driven), LTR-LGBM tends to do well when features are informative but is noisier on the long tail (illustrative only, not derived from a real run).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
N_USERS, N_ITEMS = 50, 200

# Ground truth: for each user, the items they actually interacted with (held-out).
true_relevance = {u: set(rng.choice(N_ITEMS, size=8, replace=False)) for u in range(N_USERS)}

# AryColBring-style ranking: correlated with true relevance, moderate noise.
def simulate_arycolbring_ranking(user_id: int, noise: float = 0.6) -> list:
    base_scores = np.zeros(N_ITEMS)
    base_scores[list(true_relevance[user_id])] += 1.0
    base_scores += rng.normal(scale=noise, size=N_ITEMS)
    return list(np.argsort(base_scores)[::-1])

# LTR-LGBM-style ranking: slightly stronger signal but occasional feature gaps
# (simulated as a subset of items getting a large noise penalty).
def simulate_ltrlgbm_ranking(user_id: int, noise: float = 0.45, gap_rate: float = 0.1) -> list:
    base_scores = np.zeros(N_ITEMS)
    base_scores[list(true_relevance[user_id])] += 1.1
    base_scores += rng.normal(scale=noise, size=N_ITEMS)
    gap_items = rng.choice(N_ITEMS, size=int(N_ITEMS * gap_rate), replace=False)
    base_scores[gap_items] -= 1.5  # missing/stale features for these items
    return list(np.argsort(base_scores)[::-1])

print("Simulators defined.")

## 3. Precision@K, Recall@K, NDCG@K

In [ ]:
def precision_at_k(ranking: list, relevant: set, k: int) -> float:
    top_k = ranking[:k]
    return len(set(top_k) & relevant) / k


def recall_at_k(ranking: list, relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    top_k = ranking[:k]
    return len(set(top_k) & relevant) / len(relevant)


def ndcg_at_k(ranking: list, relevant: set, k: int) -> float:
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(ranking[:k]) if item in relevant)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate(ranking_fn, k: int = 10) -> dict:
    metrics = {"precision_at_k": [], "recall_at_k": [], "ndcg_at_k": []}
    for u in range(N_USERS):
        ranking = ranking_fn(u)
        relevant = true_relevance[u]
        metrics["precision_at_k"].append(precision_at_k(ranking, relevant, k))
        metrics["recall_at_k"].append(recall_at_k(ranking, relevant, k))
        metrics["ndcg_at_k"].append(ndcg_at_k(ranking, relevant, k))
    return {name: float(np.mean(vals)) for name, vals in metrics.items()}


arycolbring_metrics = evaluate(simulate_arycolbring_ranking, k=10)
ltrlgbm_metrics     = evaluate(simulate_ltrlgbm_ranking, k=10)

comparison = pd.DataFrame({"AryColBring": arycolbring_metrics, "LTR-LGBM": ltrlgbm_metrics}).round(4)
comparison

## 4. Reading the result

On this synthetic profile, LTR-LGBM edges out AryColBring on average -- but only because we simulated it with a *slightly* stronger base signal. The `gap_rate` term (items with missing/stale features) is the more interesting part: try raising `gap_rate` in `simulate_ltrlgbm_ranking` above and re-running Section 3 -- LTR-LGBM's numbers degrade quickly once feature coverage isn't complete, while AryColBring is unaffected (it never depended on those features in the first place). That's the real trade-off in production: LTR-LGBM's ceiling is higher *when your features are good*, but it degrades in exactly the cases AryColBring doesn't care about.

## Summary

- Same evaluation harness (Precision@K / Recall@K / NDCG@K) works for both models -- they just need to produce a ranked item list per user.
- AryColBring: cheaper to serve, robust to missing features, better cold-start story via item-to-item fallback.
- LTR-LGBM: can exploit business features directly, but only as good as feature coverage/freshness.
- In practice: many production stacks use LTR-LGBM to *re-rank* AryColBring's candidate list, combining collaborative-filtering recall with feature-driven precision -- neither model needs to "win" outright.